# Daraja: Somali-Swahili Translation Model

Fine-tuning Gemma 2B for Somali ↔ Swahili translation using the NLLB parallel corpus.

**Instructions:**
1. Run Cell 1 (pip installs)
2. **RESTART KERNEL** (Runtime → Restart session)
3. Run Cell 2 onwards (DO NOT restart after this)
4. Download outputs before session ends

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# After running this cell, RESTART THE KERNEL before continuing
# ============================================================
!pip install -q peft==0.10.0 bitsandbytes>=0.43.0 accelerate
print("="*50)
print("DONE! Now restart the kernel:")
print("Runtime → Restart session")
print("Then run Cell 2")
print("="*50)

In [ ]:
# ============================================================
# CELL 2: SETUP AND LOGIN
# Run this AFTER kernel restart
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only first GPU

# Login to HuggingFace (required for Gemma)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace!")

In [ ]:
# ============================================================
# CELL 3: IMPORTS AND CONFIG
# ============================================================
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
from pathlib import Path

# Output directory
OUTPUT_DIR = Path("/kaggle/working/daraja-model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config
MODEL_NAME = "google/gemma-2-2b-it"
MAX_SEQ_LENGTH = 128
NUM_SAMPLES = 30000
BATCH_SIZE = 4
GRAD_ACCUM = 8
EPOCHS = 2
LEARNING_RATE = 2e-4

print(f"Output directory: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Samples: {NUM_SAMPLES}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ============================================================
# CELL 4: LOAD DATA
# ============================================================
print("Loading NLLB parallel corpus...")

# Update these paths to match your Kaggle dataset
DATA_DIR = Path("/kaggle/input/daraja-seed-data")  # Adjust if different
source_file = DATA_DIR / "NLLB.so-sw.so"
target_file = DATA_DIR / "NLLB.so-sw.sw"

# Check if files exist
if not source_file.exists():
    print(f"ERROR: {source_file} not found!")
    print("Available files in /kaggle/input:")
    !ls -la /kaggle/input/
    raise FileNotFoundError("Data files not found. Check your dataset input.")

with open(source_file, 'r', encoding='utf-8') as f:
    sources = [line.strip() for line in f]
with open(target_file, 'r', encoding='utf-8') as f:
    targets = [line.strip() for line in f]

print(f"Total pairs available: {len(sources):,}")

# Filter to short sentences (avoids truncation issues)
pairs = [(s, t) for s, t in zip(sources, targets) 
         if len(s) < 100 and len(t) < 100 and len(s) > 5 and len(t) > 5]
pairs = pairs[:NUM_SAMPLES]

print(f"Filtered to {len(pairs):,} short sentence pairs")

# Format for training
texts = [f"Translate Somali to Swahili:\n{s}\nSwahili:\n{t}" for s, t in pairs]

# Preview
print("\nSample training examples:")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(texts[i])

In [ ]:
# ============================================================
# CELL 5: LOAD MODEL
# ============================================================
print("Loading Gemma 2B with 4-bit quantization...")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"":0},  # Force to GPU 0
    token=hf_token,
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded! Vocab size: {len(tokenizer):,}")

In [ ]:
# ============================================================
# CELL 6: ADD LORA ADAPTERS
# ============================================================
print("Adding LoRA adapters...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ============================================================
# CELL 7: PREPARE DATASET
# ============================================================
print("Tokenizing dataset...")

dataset = Dataset.from_dict({"text": texts})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"],
    desc="Tokenizing"
)

# Add labels (same as input_ids for causal LM)
tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": x["input_ids"].copy()},
    desc="Adding labels"
)

print(f"Dataset ready: {len(tokenized_dataset):,} samples")
print(f"Columns: {tokenized_dataset.column_names}")

In [ ]:
# ============================================================
# CELL 8: TRAIN MODEL
# ============================================================
print("Setting up trainer...")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    logging_steps=100,
    save_steps=500,
    fp16=True,  # Use fp16 (T4 doesn't support bf16)
    bf16=False,
    optim="adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

total_steps = (len(tokenized_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
print(f"\nTraining configuration:")
print(f"  Samples: {len(tokenized_dataset):,}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total steps: ~{total_steps:,}")
print(f"  Estimated time: ~2 hours")
print("\n" + "="*50)
print("TRAINING STARTED")
print("="*50 + "\n")

trainer.train()

print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)

In [ ]:
# ============================================================
# CELL 9: SAVE MODEL
# ============================================================
print("Saving model...")

LORA_DIR = OUTPUT_DIR / "lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

print(f"Model saved to: {LORA_DIR}")
print("\nSaved files:")
for f in LORA_DIR.iterdir():
    size = f.stat().st_size / 1024
    print(f"  {f.name}: {size:.1f} KB")

In [ ]:
# ============================================================
# CELL 10: TEST TRANSLATIONS
# ============================================================
print("Testing translations...\n")

model.eval()

test_sentences = [
    "Waxaan rabaa in aan kaa caawiyo",  # I want to help you
    "Magaalada waa weyn tahay",          # The city is big
    "Mahadsanid",                         # Thank you
    "Nabad",                              # Peace/Hello
    "Waa maxay magacaaga?",               # What is your name?
]

print("="*60)
print("TRANSLATION RESULTS")
print("="*60)

for sent in test_sentences:
    prompt = f"Translate Somali to Swahili:\n{sent}\nSwahili:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translation = result.split("Swahili:\n")[-1].strip()
    
    print(f"\nSomali:  {sent}")
    print(f"Swahili: {translation}")

print("\n" + "="*60)

In [ ]:
# ============================================================
# CELL 11: VERIFY LORA WEIGHTS (DIAGNOSTIC)
# ============================================================
print("Verifying LoRA weights learned something...\n")

lora_params = [(n, p) for n, p in model.named_parameters() if 'lora' in n.lower()]
print(f"Total LoRA parameters: {len(lora_params)}")

print("\nSample LoRA weight statistics:")
for name, param in lora_params[:4]:
    mean = param.data.float().mean().item()
    std = param.data.float().std().item()
    print(f"  {name.split('.')[-3]}_{name.split('.')[-1]}: mean={mean:.6f}, std={std:.6f}")

print("\nIf std values are much larger than 0.01, the model learned!")

In [ ]:
# ============================================================
# CELL 12: PUSH TO HUGGINGFACE HUB (OPTIONAL BUT RECOMMENDED)
# This ensures your model persists after the Kaggle session ends
# ============================================================
PUSH_TO_HUB = False  # Set to True to push
HUB_REPO_NAME = "daraja-somali-swahili"  # Change to your preferred name

if PUSH_TO_HUB:
    print(f"Pushing model to HuggingFace Hub as '{HUB_REPO_NAME}'...")
    model.push_to_hub(HUB_REPO_NAME, token=hf_token, private=True)
    tokenizer.push_to_hub(HUB_REPO_NAME, token=hf_token, private=True)
    print(f"Model pushed! View at: https://huggingface.co/{HUB_REPO_NAME}")
else:
    print("Skipping HuggingFace Hub push.")
    print("Set PUSH_TO_HUB = True to save your model permanently.")

In [ ]:
# ============================================================
# CELL 13: LIST OUTPUT FILES FOR DOWNLOAD
# ============================================================
print("Output files available for download:")
print("="*50)

import os
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{subindent}{file} ({size_mb:.2f} MB)")

print("\n" + "="*50)
print("IMPORTANT: Download these files before your session ends!")
print("Go to: Output tab → Download All")
print("="*50)

## Next Steps

After training:
1. **Download the LoRA weights** from `/kaggle/working/daraja-model/lora/`
2. **Or push to HuggingFace Hub** (recommended for persistence)
3. **Convert to GGUF** for Ollama deployment (see next notebook)
4. **Run evaluation** to get BLEU/chrF++ scores